In [ ]:
# !pip install shap

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# ColumnTransformer is for the use of OneHotEncoder
from sklearn.compose import ColumnTransformer

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTENC
import shap
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [ ]:
''' 雖然 normalization 對於 random forest and xgboost 好像沒幫助 不會增加效能 但是還是跑看看差多少 '''
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv('./raw_data/raw_data.csv')

In [ ]:
def summarize_fold_reports(fold_reports, label='1'):
    """平均每個 fold 的 precision、recall、f1-score（針對指定的 label）"""
    precisions = [report[label]['precision'] for report in fold_reports]
    recalls = [report[label]['recall'] for report in fold_reports]
    f1s = [report[label]['f1-score'] for report in fold_reports]

    return pd.Series({
        'Precision': np.mean(precisions),
        'Recall': np.mean(recalls),
        'F1-score': np.mean(f1s)
    })

# Raw dataset analysis

In [ ]:
data = df

In [ ]:
print(f"資料維度: {data.shape}")
print(data.info())

In [ ]:
# 3.1 缺失值檢查
missing = data.isnull().sum()
print(missing[missing > 0])

In [ ]:
# 3.2 目標變數分布 (假設目標欄位為 'recurrent_stroke')
import seaborn as sns

sns.countplot(data=data, x='Second_Stroke')
plt.title('Recurrent Stroke Distribution')
plt.show()

## 底下這個是會有data leakage版本的跑法（recall and precision都會是1）

In [ ]:
# # --- Step 0: Import & Prepare ---
# from sklearn.model_selection import StratifiedKFold
# from sklearn.preprocessing import OneHotEncoder, StandardScaler
# from sklearn.compose import ColumnTransformer
# from sklearn.metrics import classification_report
# from imblearn.over_sampling import SMOTENC
# from sklearn.ensemble import RandomForestClassifier
# from xgboost import XGBClassifier
# import shap
# import numpy as np
# import pandas as pd
# from sklearn.utils import shuffle

# # --- Step 1: Define your original features ---
# categorical_cols = ['sex', 'tPA(0/1)', 'EVT(0/1)', 'HTN(0/1)', 'DM(0/1)',
#                      'Dyslipidemia(0/1)', 'Af(0/1)', 'smoking(Y/N/Q)', 'MRS']
# num_cols = [c for c in data.columns if c not in categorical_cols + ['Second_Stroke']]

# # --- Step 2: Split into pos/neg and cut neg into 10 folds ---
# pos_data = data[data['Second_Stroke'] == 1].reset_index(drop=True)
# neg_data = data[data['Second_Stroke'] == 0].reset_index(drop=True)
# neg_data = shuffle(neg_data, random_state=42).reset_index(drop=True)
# neg_folds = np.array_split(neg_data, 10)

# # --- Step 3: Define model dictionary ---
# models = {
#     'RandomForest': RandomForestClassifier(class_weight='balanced', random_state=42),
#     'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
# }

# # --- Step 4: Train per fold (1/10 neg + all pos) ---
# all_fold_reports = {}
# shap_values_dict = {}

# for model_name, model in models.items():
#     print(f"\nModel: {model_name}")
#     fold_reports = []

#     for i, neg_subset in enumerate(neg_folds):
#         fold_df = pd.concat([neg_subset, pos_data], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)
#         X = fold_df.drop('Second_Stroke', axis=1)
#         y = fold_df['Second_Stroke']

#         preprocessor = ColumnTransformer([
#             ('cat', OneHotEncoder(drop='first'), categorical_cols),
#             ('num', StandardScaler(), num_cols)
#         ])

#         X_processed = preprocessor.fit_transform(X)
#         cat_encoder = preprocessor.named_transformers_['cat']
#         cat_feature_count = len(cat_encoder.get_feature_names_out())
#         cat_indices = list(range(cat_feature_count))

#         smote_nc = SMOTENC(categorical_features=cat_indices, random_state=42)
#         X_resampled, y_resampled = smote_nc.fit_resample(X_processed, y)

#         if model_name == 'XGBoost':
#             scale = sum(y == 0) / sum(y == 1)
#             model.set_params(scale_pos_weight=scale)

#         model.fit(X_resampled, y_resampled)
#         y_pred = model.predict(X_processed)
#         report = classification_report(y, y_pred, output_dict=True)
#         fold_reports.append(report)

#     all_fold_reports[model_name] = fold_reports

#     # SHAP Analysis
#     explainer = shap.TreeExplainer(model)
#     shap_values = explainer.shap_values(X_resampled)
#     shap_values_dict[model_name] = (explainer, shap_values)
#     print(f"\nSHAP summary plot for {model_name}:")
#     shap.summary_plot(shap_values, X_resampled, feature_names=preprocessor.get_feature_names_out())

# # --- Summary Helper ---
# def summarize_fold_reports(fold_reports, label='1'):
#     precisions = [r[label]['precision'] for r in fold_reports]
#     recalls = [r[label]['recall'] for r in fold_reports]
#     f1s = [r[label]['f1-score'] for r in fold_reports]
#     return pd.Series({
#         'Precision': np.mean(precisions),
#         'Recall': np.mean(recalls),
#         'F1-score': np.mean(f1s)
#     })

# # Print results
# for name, reports in all_fold_reports.items():
#     print(f"\n{name} average performance:")
#     print(summarize_fold_reports(reports))


In [ ]:
# Step 2: Encode categorical features

categorical_cols = ['sex', 'tPA(0/1)','EVT(0/1)','HTN(0/1)','DM(0/1)','Dyslipidemia(0/1)','Af(0/1)','smoking(Y/N/Q)', 'MRS']
num_cols = [c for c in data.columns if c not in categorical_cols + ['Second_Stroke']]


# label_encoders = {}
# for col in categorical_cols:
#     le = LabelEncoder()
#     data[col] = le.fit_transform(data[col])
#     label_encoders[col] = le

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first'), categorical_cols),
        ('num', 'passthrough', num_cols)
    ]
)


X = data.drop('Second_Stroke', axis=1)
y = data['Second_Stroke']
categorical_feature_indices = [X.columns.get_loc(col) for col in categorical_cols]

# Step 3: Stratified 10-fold CV
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

num_neg = sum(y == 0)
num_pos = sum(y == 1)
scale = num_neg/num_pos


models = {
    'RandomForest': RandomForestClassifier(class_weight = 'balanced', random_state=42),
    'XGBoost': XGBClassifier(scale_pos_weight = scale, use_label_encoder=False, eval_metric='logloss', random_state=42) #,
#     'SVM': SVC(probability=True, random_state=42)
}

# Store SHAP values for each model
shap_values_dict = {}
all_fold_reports = {}

for model_name, model in models.items():
    print(f"\nModel: {model_name}")
    fold_reports = []
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        

        
        # OneHotEncoder - Transform
        X_train = preprocessor.fit_transform(X_train)
        X_test = preprocessor.transform(X_test)

        
        '''
        原本有想把底下standardize之後的 variable name 改成 X_trian_scaled and X_test_scaled 
        但這樣後面就要全改 沒關係就先這樣好了 要改之後再說
        '''
        
        # Standardization
        scaler = StandardScaler()
        
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        
        
        
        

        smote_nc = SMOTENC(categorical_features=categorical_feature_indices, random_state=42)
        X_resampled, y_resampled = smote_nc.fit_resample(X_train, y_train)

        model.fit(X_resampled, y_resampled)
        y_pred = model.predict(X_test)
        report = classification_report(y_test, y_pred, output_dict=True)
        fold_reports.append(report)

    all_fold_reports[model_name] = fold_reports
        
    # Train final model for SHAP visualization on the full (resampled) training data
    X_resampled_full, y_resampled_full = smote_nc.fit_resample(X, y)
    model.fit(X_resampled_full, y_resampled_full)

    # SHAP analysis (tree-based explainer for RF and XGB, kernel explainer for SVM)
    if model_name in ['RandomForest', 'XGBoost']:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_resampled_full)
        shap_values_dict[model_name] = (explainer, shap_values)
        print(f"\nSHAP summary plot (complete dataset) for {model_name}:")
        shap.summary_plot(shap_values, X_resampled_full, feature_names=X.columns)
        
    else:
        explainer = shap.KernelExplainer(model.predict_proba, shap.sample(X_resampled_full, 50))
        shap_values = explainer.shap_values(shap.sample(X_resampled_full, 50))
#       shap_values = explainer.shap_values(X_resampled_full)
        shap_values_dict[model_name] = (explainer, shap_values)

        # SHAP summary plot
        print(f"\nSHAP summary plot (complete dataset) for {model_name}:")
        shap.summary_plot(shap_values, shap.sample(X_resampled_full, 50), feature_names=X.columns)
#         shap.summary_plot(shap_values, X_resampled_full, feature_names=X.columns)

# Done!

In [ ]:
rf_metrics = summarize_fold_reports(all_fold_reports['RandomForest'])
xgb_metrics = summarize_fold_reports(all_fold_reports['XGBoost'])

In [ ]:
performance_df = pd.DataFrame({
    'RandomForest': rf_metrics,
    'XGBoost': xgb_metrics
})
print(performance_df)

# Preparing male and female datasets for independent analyses

In [ ]:
male_data = df[df['sex'] == 0].reset_index(drop=True)
female_data = df[df['sex'] == 1].reset_index(drop=True)

# Male data analysis and model training (& SHAP plotting)

In [ ]:
data = male_data

In [ ]:
print(f"資料維度: {data.shape}")
print(data.info())

In [ ]:
# 3.1 缺失值檢查
missing = data.isnull().sum()
print(missing[missing > 0])

In [ ]:
# 3.2 目標變數分布 (假設目標欄位為 'recurrent_stroke')
import seaborn as sns

sns.countplot(data=data, x='Second_Stroke')
plt.title('Recurrent Stroke Distribution')
plt.show()

In [ ]:
data.head()

In [ ]:
data = data.drop('sex', axis=1)

## 底下這個是會有data leakage版本的跑法（recall and precision都會是1）

In [ ]:
# # --- Step 0: Import & Prepare ---
# from sklearn.model_selection import StratifiedKFold
# from sklearn.preprocessing import OneHotEncoder, StandardScaler
# from sklearn.compose import ColumnTransformer
# from sklearn.metrics import classification_report
# from imblearn.over_sampling import SMOTENC
# from sklearn.ensemble import RandomForestClassifier
# from xgboost import XGBClassifier
# import shap
# import numpy as np
# import pandas as pd
# from sklearn.utils import shuffle

# # --- Step 1: Define your original features ---
# categorical_cols = ['tPA(0/1)', 'EVT(0/1)', 'HTN(0/1)', 'DM(0/1)',
#                      'Dyslipidemia(0/1)', 'Af(0/1)', 'smoking(Y/N/Q)', 'MRS']
# num_cols = [c for c in data.columns if c not in categorical_cols + ['Second_Stroke']]

# # --- Step 2: Split into pos/neg and cut neg into 10 folds ---
# pos_data = data[data['Second_Stroke'] == 1].reset_index(drop=True)
# neg_data = data[data['Second_Stroke'] == 0].reset_index(drop=True)
# neg_data = shuffle(neg_data, random_state=42).reset_index(drop=True)
# neg_folds = np.array_split(neg_data, 10)

# # --- Step 3: Define model dictionary ---
# models = {
#     'RandomForest': RandomForestClassifier(class_weight='balanced', random_state=42),
#     'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
# }

# # --- Step 4: Train per fold (1/10 neg + all pos) ---
# all_fold_reports = {}
# shap_values_dict = {}

# for model_name, model in models.items():
#     print(f"\nModel: {model_name}")
#     fold_reports = []

#     for i, neg_subset in enumerate(neg_folds):
#         fold_df = pd.concat([neg_subset, pos_data], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)
#         X = fold_df.drop('Second_Stroke', axis=1)
#         y = fold_df['Second_Stroke']

#         preprocessor = ColumnTransformer([
#             ('cat', OneHotEncoder(drop='first'), categorical_cols),
#             ('num', StandardScaler(), num_cols)
#         ])

#         X_processed = preprocessor.fit_transform(X)
#         cat_encoder = preprocessor.named_transformers_['cat']
#         cat_feature_count = len(cat_encoder.get_feature_names_out())
#         cat_indices = list(range(cat_feature_count))

#         smote_nc = SMOTENC(categorical_features=cat_indices, random_state=42)
#         X_resampled, y_resampled = smote_nc.fit_resample(X_processed, y)

#         if model_name == 'XGBoost':
#             scale = sum(y == 0) / sum(y == 1)
#             model.set_params(scale_pos_weight=scale)

#         model.fit(X_resampled, y_resampled)
#         y_pred = model.predict(X_processed)
#         report = classification_report(y, y_pred, output_dict=True)
#         fold_reports.append(report)

#     all_fold_reports[model_name] = fold_reports

#     # SHAP Analysis
#     explainer = shap.TreeExplainer(model)
#     shap_values = explainer.shap_values(X_resampled)
#     shap_values_dict[model_name] = (explainer, shap_values)
#     print(f"\nSHAP summary plot for {model_name}:")
#     shap.summary_plot(shap_values, X_resampled, feature_names=preprocessor.get_feature_names_out())

# # --- Summary Helper ---
# def summarize_fold_reports(fold_reports, label='1'):
#     precisions = [r[label]['precision'] for r in fold_reports]
#     recalls = [r[label]['recall'] for r in fold_reports]
#     f1s = [r[label]['f1-score'] for r in fold_reports]
#     return pd.Series({
#         'Precision': np.mean(precisions),
#         'Recall': np.mean(recalls),
#         'F1-score': np.mean(f1s)
#     })

# # Print results
# for name, reports in all_fold_reports.items():
#     print(f"\n{name} average performance:")
#     print(summarize_fold_reports(reports))


In [ ]:
# Step 2: Encode categorical features

categorical_cols = ['tPA(0/1)','EVT(0/1)','HTN(0/1)','DM(0/1)','Dyslipidemia(0/1)','Af(0/1)','smoking(Y/N/Q)', 'MRS']
num_cols = [c for c in data.columns if c not in categorical_cols + ['Second_Stroke']]


# label_encoders = {}
# for col in categorical_cols:
#     le = LabelEncoder()
#     data[col] = le.fit_transform(data[col])
#     label_encoders[col] = le

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first'), categorical_cols),
        ('num', 'passthrough', num_cols)
    ]
)

    
X = data.drop('Second_Stroke', axis=1)
y = data['Second_Stroke']
categorical_feature_indices = [X.columns.get_loc(col) for col in categorical_cols]

# Step 3: Stratified 10-fold CV
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

num_neg = sum(y == 0)
num_pos = sum(y == 1)
scale = num_neg/num_pos


models = {
    'RandomForest': RandomForestClassifier(class_weight = 'balanced', random_state=42),
    'XGBoost': XGBClassifier(scale_pos_weight = scale, use_label_encoder=False, eval_metric='logloss', random_state=42) #,
#     'SVM': SVC(probability=True, random_state=42)
}

# Store SHAP values for each model
shap_values_dict = {}
all_fold_reports = {}

for model_name, model in models.items():
    print(f"\nModel: {model_name}")
    fold_reports = []
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        
        # OneHotEncoder - Transform
        X_train = preprocessor.fit_transform(X_train)
        X_test = preprocessor.transform(X_test)

        
        '''
        原本有想把底下standardize之後的 variable name 改成 X_trian_scaled and X_test_scaled 
        但這樣後面就要全改 沒關係就先這樣好了 要改之後再說
        '''
        
        # Standardization
        scaler = StandardScaler()
        
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        
        
        
        smote_nc = SMOTENC(categorical_features=categorical_feature_indices, random_state=42)
        X_resampled, y_resampled = smote_nc.fit_resample(X_train, y_train)

        model.fit(X_resampled, y_resampled)
        y_pred = model.predict(X_test)
        report = classification_report(y_test, y_pred, output_dict=True)
        fold_reports.append(report)

    all_fold_reports[model_name] = fold_reports
        
    # Train final model for SHAP visualization on the full (resampled) training data
    X_resampled_full, y_resampled_full = smote_nc.fit_resample(X, y)
    model.fit(X_resampled_full, y_resampled_full)

    # SHAP analysis (tree-based explainer for RF and XGB, kernel explainer for SVM)
    if model_name in ['RandomForest', 'XGBoost']:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_resampled_full)
        shap_values_dict[model_name] = (explainer, shap_values)
        print(f"\nSHAP summary plot (male) for {model_name}:")
        shap.summary_plot(shap_values, X_resampled_full, feature_names=X.columns)
        
    else:
        explainer = shap.KernelExplainer(model.predict_proba, shap.sample(X_resampled_full, 50))
        shap_values = explainer.shap_values(shap.sample(X_resampled_full, 50))
#       shap_values = explainer.shap_values(X_resampled_full)
        shap_values_dict[model_name] = (explainer, shap_values)

        # SHAP summary plot
        print(f"\nSHAP summary plot (male) for {model_name}:")
        shap.summary_plot(shap_values, shap.sample(X_resampled_full, 50), feature_names=X.columns)
#         shap.summary_plot(shap_values, X_resampled_full, feature_names=X.columns)

# Done!

In [ ]:
# def summarize_fold_reports(fold_reports, label='1'):
#     """平均每個 fold 的 precision、recall、f1-score（針對指定的 label）"""
#     precisions = [report[label]['precision'] for report in fold_reports]
#     recalls = [report[label]['recall'] for report in fold_reports]
#     f1s = [report[label]['f1-score'] for report in fold_reports]

#     return pd.Series({
#         'Precision': np.mean(precisions),
#         'Recall': np.mean(recalls),
#         'F1-score': np.mean(f1s)
#     })

In [ ]:
rf_metrics = summarize_fold_reports(all_fold_reports['RandomForest'])
xgb_metrics = summarize_fold_reports(all_fold_reports['XGBoost'])

In [ ]:
performance_df = pd.DataFrame({
    'RandomForest': rf_metrics,
    'XGBoost': xgb_metrics
})
print(performance_df)

# Female data analysis and model training (& SHAP plotting)

In [ ]:
data = female_data
print(f"資料維度: {data.shape}")
print(data.info())

In [ ]:
# 3.1 缺失值檢查
missing = data.isnull().sum()
print(missing[missing > 0])

In [ ]:
# 3.2 目標變數分布 (假設目標欄位為 'recurrent_stroke')
import seaborn as sns

sns.countplot(data=data, x='Second_Stroke')
plt.title('Recurrent Stroke Distribution')
plt.show()

In [ ]:
data = data.drop('sex', axis=1)

In [ ]:
# Step 2: Encode categorical features


categorical_cols = ['tPA(0/1)','EVT(0/1)','HTN(0/1)','DM(0/1)','Dyslipidemia(0/1)','Af(0/1)','smoking(Y/N/Q)', 'MRS']
num_cols = [c for c in data.columns if c not in categorical_cols + ['Second_Stroke']]


label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    label_encoders[col] = le

# preprocessor = ColumnTransformer(
#     transformers=[
#         ('cat', OneHotEncoder(drop='first'), categorical_cols),
#         ('num', 'passthrough', num_cols)
#     ]
# )


X = data.drop('Second_Stroke', axis=1)
y = data['Second_Stroke']
categorical_feature_indices = [X.columns.get_loc(col) for col in categorical_cols]

# Step 3: Stratified 10-fold CV
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

num_neg = sum(y == 0)
num_pos = sum(y == 1)
scale = num_neg/num_pos


models = {
    'RandomForest': RandomForestClassifier(class_weight = 'balanced', random_state=42),
    'XGBoost': XGBClassifier(scale_pos_weight = scale, use_label_encoder=False, eval_metric='logloss', random_state=42) #,
#     'SVM': SVC(probability=True, random_state=42)
}

# Store SHAP values for each model
shap_values_dict = {}
all_fold_reports = {}

for model_name, model in models.items():
    print(f"\nModel: {model_name}")
    fold_reports = []
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        
        
        # OneHotEncoder - Transform
#         X_train = preprocessor.fit_transform(X_train)
#         X_test = preprocessor.transform(X_test)

        
        '''
        原本有想把底下standardize之後的 variable name 改成 X_trian_scaled and X_test_scaled 
        但這樣後面就要全改 沒關係就先這樣好了 要改之後再說
        '''
        
        # Standardization
#         scaler = StandardScaler()
        
#         X_train = scaler.fit_transform(X_train)
#         X_test = scaler.transform(X_test)
        
        
        
        smote_nc = SMOTENC(categorical_features=categorical_feature_indices, random_state=42)
        X_resampled, y_resampled = smote_nc.fit_resample(X_train, y_train)

        model.fit(X_resampled, y_resampled)
        y_pred = model.predict(X_test)
        report = classification_report(y_test, y_pred, output_dict=True)
        fold_reports.append(report)
    
    all_fold_reports[model_name] = fold_reports

#     print("fold_reports: ", fold_reports)
    # Train final model for SHAP visualization on the full (resampled) training data
    X_resampled_full, y_resampled_full = smote_nc.fit_resample(X, y)
    model.fit(X_resampled_full, y_resampled_full)

    # SHAP analysis (tree-based explainer for RF and XGB, kernel explainer for SVM)
    if model_name in ['RandomForest', 'XGBoost']:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_resampled_full)
        shap_values_dict[model_name] = (explainer, shap_values)
        print(f"\nSHAP summary plot (female) for {model_name}:")
        shap.summary_plot(shap_values, X_resampled_full, feature_names=X.columns)
        
    else:
        explainer = shap.KernelExplainer(model.predict_proba, shap.sample(X_resampled_full, 50))
        shap_values = explainer.shap_values(shap.sample(X_resampled_full, 50))
#       shap_values = explainer.shap_values(X_resampled_full)
        shap_values_dict[model_name] = (explainer, shap_values)

        # SHAP summary plot
        print(f"\nSHAP summary plot (female) for {model_name}:")
        shap.summary_plot(shap_values, shap.sample(X_resampled_full, 50), feature_names=X.columns)
#         shap.summary_plot(shap_values, X_resampled_full, feature_names=X.columns)

# Done!

In [ ]:
rf_metrics = summarize_fold_reports(all_fold_reports['RandomForest'])
xgb_metrics = summarize_fold_reports(all_fold_reports['XGBoost'])

In [ ]:
performance_df = pd.DataFrame({
    'RandomForest': rf_metrics,
    'XGBoost': xgb_metrics
})
print(performance_df)